In [ ]:
!pip install segmentation-models-pytorch rasterio albumentations torch tifffile numpy tqdm

# LinkNET Orthomosaic Inference — Sliding Window

This notebook runs **LinkNET** segmentation inference on large orthomosaics using a **sliding window** strategy.

Overlapping tile predictions are blended with **Gaussian weighting** before thresholding, which eliminates seam artefacts at tile boundaries.

---
**Pipeline overview**
1. Install dependencies  
2. Imports & configuration  
3. Model definition & loading  
4. Orthomosaic loading & normalisation  
5. Tile helpers  
6. Sliding-window inference  
7. Save outputs  
8. Visualisation  
9. Run everything  

In [ ]:

import os
import gc
import argparse
import warnings
import numpy as np
import tifffile
import torch
import torch.nn.functional as F
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm

try:
    import rasterio
    from rasterio.transform import from_bounds
    RASTERIO_AVAILABLE = True
except ImportError:
    RASTERIO_AVAILABLE = False

warnings.filterwarnings("ignore")

In [ ]:
# ---------------------------------------------------------------------------
# Configuration — keep in sync with training CFG
# ---------------------------------------------------------------------------
class CFG:
    model_name  = "Linknet"
    encoder     = "efficientnet-b7"
    num_classes = 2
    img_size    = [512, 512]
    size        = 512
    device      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Normalisation stats used during training (DroneDataset)
    mean = [0.4855, 0.5464, 0.4754]
    std  = [0.1446, 0.1593, 0.1308]

print(f"[INFO] Running on : {CFG.device}")

[INFO] Running on : cuda


## Model — Build & Load Checkpoint

In [ ]:
def build_model() -> smp.Linknet:
    """Instantiate LinkNET with the same architecture used during training."""
    model = smp.Linknet(
        encoder_name          = CFG.encoder,
        encoder_weights       = None,   # weights come from the checkpoint
        encoder_depth         = 5,
        classes               = CFG.num_classes,
        activation            = "sigmoid",
        decoder_use_batchnorm = True,
    )
    return model


def load_model(checkpoint_path: str) -> smp.Linknet:
    """Load a saved LinkNET checkpoint and set to eval mode."""
    print(f"[INFO] Loading model from: {checkpoint_path}")
    model = build_model()
    state = torch.load(checkpoint_path, map_location=CFG.device)

    # Support raw state-dict saves and wrapped {'model'/'state_dict': ...} dicts
    if isinstance(state, dict) and "model" in state:
        state = state["model"]
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]

    model.load_state_dict(state)
    model.to(CFG.device)
    model.eval()
    print(f"[INFO] Model loaded on {CFG.device}")
    return model

## Orthomosaic Loading & Normalisation

In [ ]:
def _load_tifffile(path: str) -> np.ndarray:
    img = tifffile.imread(path)
    print(f"[INFO] Loaded via tifffile — shape: {img.shape}, dtype: {img.dtype}")
    return img


def _normalise_to_uint8(img: np.ndarray) -> np.ndarray:
    """Convert any numeric dtype/layout to uint8 RGB (H, W, 3)."""
    # Handle channel-first (C, H, W)
    if img.ndim == 3 and img.shape[0] in (1, 3, 4) and img.shape[0] < img.shape[1]:
        img = np.moveaxis(img, 0, -1)

    # Drop alpha channel
    if img.ndim == 3 and img.shape[2] == 4:
        img = img[:, :, :3]

    # Single-channel → 3-channel
    if img.ndim == 2:
        img = np.stack([img, img, img], axis=2)
    elif img.ndim == 3 and img.shape[2] == 1:
        img = np.repeat(img, 3, axis=2)

    img = img.astype(np.float32)

    if img.max() <= 1.0:
        img = img * 255.0
    else:
        lo, hi = img.min(), img.max()
        if hi > lo:
            img = (img - lo) / (hi - lo) * 255.0

    return np.clip(img, 0, 255).astype(np.uint8)


def load_orthomosaic(path: str):
    """
    Load an orthomosaic from a GeoTIFF or TIFF file.

    Returns
    -------
    image : np.ndarray  (H, W, 3) uint8
    meta  : dict        geo-metadata profile (empty if rasterio unavailable)
    """
    meta = {}

    if RASTERIO_AVAILABLE and path.lower().endswith((".tif", ".tiff")):
        try:
            with rasterio.open(path) as src:
                meta = src.profile
                n_bands = min(src.count, 3)
                img = src.read(list(range(1, n_bands + 1)))   # (C, H, W)
                img = np.moveaxis(img, 0, -1)                  # (H, W, C)
                if n_bands < 3:
                    img = np.repeat(img, 3 // n_bands, axis=2)[:, :, :3]
            print(f"[INFO] Loaded via rasterio — shape: {img.shape}, dtype: {img.dtype}")
        except Exception as e:
            print(f"[WARN] rasterio failed ({e}), falling back to tifffile")
            img = _load_tifffile(path)
    else:
        img = _load_tifffile(path)

    img = _normalise_to_uint8(img)
    print(f"[INFO] Final orthomosaic shape: {img.shape}")
    return img, meta

## Tile Helpers

In [ ]:
def get_tiles(h: int, w: int, tile_size: int, overlap: int):
    """
    Yield (row_start, row_end, col_start, col_end) for every sliding-window
    position that fully covers the (h, w) image.
    Edge tiles are anchored to the image boundary so no pixels are missed.
    """
    stride = tile_size - overlap
    tiles  = []

    r = 0
    while r < h:
        r_end   = min(r + tile_size, h)
        r_start = max(r_end - tile_size, 0)

        c = 0
        while c < w:
            c_end   = min(c + tile_size, w)
            c_start = max(c_end - tile_size, 0)
            tiles.append((r_start, r_end, c_start, c_end))
            if c_end == w:
                break
            c += stride

        if r_end == h:
            break
        r += stride

    return tiles


def preprocess_tile(tile: np.ndarray, transform: A.Compose) -> torch.Tensor:
    """Apply albumentations normalisation pipeline; return (1, C, H, W) tensor."""
    return transform(image=tile)["image"].unsqueeze(0)


def _gaussian_weight(h: int, w: int, sigma_ratio: float = 0.35) -> np.ndarray:
    """
    2-D Gaussian centred on the tile.
    Centre pixels get higher weight → reduces seam artefacts at tile edges.
    """
    cy, cx    = h / 2.0, w / 2.0
    sigma_y   = cy * sigma_ratio
    sigma_x   = cx * sigma_ratio
    y         = np.arange(h, dtype=np.float32)
    x         = np.arange(w, dtype=np.float32)
    yy, xx    = np.meshgrid(y, x, indexing="ij")
    weight    = np.exp(-0.5 * (((yy - cy) / sigma_y) ** 2 + ((xx - cx) / sigma_x) ** 2))
    return weight.astype(np.float32)

## Sliding-Window Inference

Tiles are batched and run through the model. Softmax class-1 probabilities are
accumulated into a probability map using Gaussian weighting, then thresholded
into the final binary mask.

In [ ]:
def sliding_window_inference(
    model:      torch.nn.Module,
    image:      np.ndarray,
    tile_size:  int   = 512,
    overlap:    int   = 64,
    batch_size: int   = 4,
    threshold:  float = 0.5,
):
    """
    Run LinkNET inference on a large orthomosaic using a sliding window.

    Parameters
    ----------
    model      : trained LinkNET in eval mode
    image      : (H, W, 3) uint8 orthomosaic
    tile_size  : side length of each square tile — must match training size
    overlap    : pixel overlap between adjacent tiles (larger → smoother seams)
    batch_size : tiles processed per GPU forward pass
    threshold  : probability cut-off for the foreground (seagrass) class

    Returns
    -------
    pred_mask : (H, W) uint8  — 0 = background, 1 = seagrass
    prob_map  : (H, W) float32 — averaged foreground probabilities
    """
    h, w = image.shape[:2]
    print(f"[INFO] Orthomosaic : {h} x {w} px")
    print(f"[INFO] Tile {tile_size}px | Overlap {overlap}px | Batch {batch_size}")

    transform = A.Compose([
        A.Resize(tile_size, tile_size),
        A.Normalize(mean=CFG.mean, std=CFG.std),
        ToTensorV2(),
    ])

    # Accumulators for soft-vote blending
    prob_map  = np.zeros((h, w), dtype=np.float32)
    count_map = np.zeros((h, w), dtype=np.float32)

    tiles = get_tiles(h, w, tile_size, overlap)
    print(f"[INFO] Total tiles : {len(tiles)}")

    for batch_start in tqdm(range(0, len(tiles), batch_size), desc="Inference"):
        batch_tiles = tiles[batch_start : batch_start + batch_size]

        tensors, coords = [], []
        for (r0, r1, c0, c1) in batch_tiles:
            tile = image[r0:r1, c0:c1]

            # Pad edge tiles to tile_size with reflection padding
            ph = tile_size - tile.shape[0]
            pw = tile_size - tile.shape[1]
            if ph > 0 or pw > 0:
                tile = np.pad(tile, ((0, ph), (0, pw), (0, 0)), mode="reflect")

            tensors.append(preprocess_tile(tile, transform))
            coords.append((r0, r1, c0, c1))

        batch_tensor = torch.cat(tensors, dim=0).to(CFG.device)   # (B, C, H, W)

        with torch.no_grad():
            logits = model(batch_tensor)                            # (B, num_classes, H, W)
            probs  = F.softmax(logits, dim=1)[:, 1, :, :].cpu().numpy()  # (B, H, W)

        for i, (r0, r1, c0, c1) in enumerate(coords):
            tile_h, tile_w = r1 - r0, c1 - c0
            pred_tile = probs[i][:tile_h, :tile_w]   # crop padding if any

            weight = _gaussian_weight(tile_h, tile_w)
            prob_map[r0:r1, c0:c1]  += pred_tile * weight
            count_map[r0:r1, c0:c1] += weight

    count_map = np.maximum(count_map, 1e-6)
    avg_probs = prob_map / count_map
    pred_mask = (avg_probs >= threshold).astype(np.uint8)

    print(f"[INFO] Foreground pixels : {pred_mask.sum():,} / {pred_mask.size:,} "
          f"({100 * pred_mask.mean():.2f}%)")
    return pred_mask, avg_probs

## Save Outputs

In [ ]:
def save_mask(mask: np.ndarray, output_path: str, meta: dict, source_path: str = "") -> None:
    """
    Save the binary prediction mask.
    • GeoTIFF with preserved CRS/transform if rasterio is available.
    • Plain TIFF otherwise.
    """
    os.makedirs(os.path.dirname(os.path.abspath(output_path)), exist_ok=True)

    if RASTERIO_AVAILABLE and meta:
        out_meta = meta.copy()
        out_meta.update({"count": 1, "dtype": "uint8", "compress": "lzw", "nodata": None})
        with rasterio.open(output_path, "w", **out_meta) as dst:
            dst.write(mask[np.newaxis, :, :])   # (1, H, W)
        print(f"[INFO] Geo-referenced mask saved → {output_path}")
    else:
        tifffile.imwrite(output_path, mask)
        print(f"[INFO] Mask saved → {output_path}")


def save_probability_map(prob_map: np.ndarray, output_path: str, meta: dict) -> None:
    """Optionally save the raw float32 probability map alongside the binary mask."""
    prob_path = output_path.replace(".tif", "_probmap.tif")
    if RASTERIO_AVAILABLE and meta:
        out_meta = meta.copy()
        out_meta.update({"count": 1, "dtype": "float32", "compress": "lzw"})
        with rasterio.open(prob_path, "w", **out_meta) as dst:
            dst.write(prob_map[np.newaxis, :, :])
    else:
        tifffile.imwrite(prob_path, prob_map)
    print(f"[INFO] Probability map saved → {prob_path}")

## Visualisation

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


def visualise_result(
    image:     np.ndarray,
    pred_mask: np.ndarray,
    prob_map:  np.ndarray = None,
    alpha:     float      = 0.4,
    save_path: str        = "overlay.png",
) -> None:
    """
    Display and save a 3-panel (or 4-panel) QC figure:
        Orthomosaic | Prediction | Overlay | Probability Map (optional)
    Large images are downscaled to 2048 px on the longest side for display.
    """
    import cv2

    max_px = 2048
    h, w   = image.shape[:2]
    scale  = min(max_px / h, max_px / w, 1.0)

    if scale < 1.0:
        dsize      = (int(w * scale), int(h * scale))
        image_disp = cv2.resize(image,                       dsize, interpolation=cv2.INTER_AREA)
        mask_disp  = cv2.resize(pred_mask.astype(np.uint8),  dsize, interpolation=cv2.INTER_NEAREST)
        prob_disp  = cv2.resize(prob_map,                    dsize, interpolation=cv2.INTER_AREA) if prob_map is not None else None
    else:
        image_disp = image
        mask_disp  = pred_mask
        prob_disp  = prob_map

    overlay = image_disp.copy().astype(np.float32) / 255.0
    fg = mask_disp == 1
    overlay[fg, 0] = overlay[fg, 0] * (1 - alpha)
    overlay[fg, 1] = overlay[fg, 1] * (1 - alpha) + alpha
    overlay[fg, 2] = overlay[fg, 2] * (1 - alpha)

    n_panels = 4 if prob_disp is not None else 3
    fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, 6))

    axes[0].imshow(image_disp);                       axes[0].set_title("Orthomosaic");  axes[0].axis("off")
    axes[1].imshow(mask_disp, cmap="Greens");         axes[1].set_title("Prediction");   axes[1].axis("off")
    axes[2].imshow(np.clip(overlay, 0, 1));           axes[2].set_title("Overlay");      axes[2].axis("off")
    axes[2].legend(handles=[mpatches.Patch(color="lime", label="Seagrass")],
                   loc="lower right", fontsize=10)

    if prob_disp is not None:
        im = axes[3].imshow(prob_disp, cmap="viridis", vmin=0, vmax=1)
        axes[3].set_title("Probability Map");         axes[3].axis("off")
        plt.colorbar(im, ax=axes[3], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"[INFO] Overlay saved → {save_path}")

## Run Inference

Set the paths and parameters below, then run this cell.

In [ ]:
# ============================================================
#  ✏️  PARAMETERS
# ============================================================
MODEL_PATH   = "/content/drive/MyDrive/BlueCARES_Seagrass/segmentation/training_results/best_model_epoch_038.pth"          # trained checkpoint
INPUT_PATH   = "/content/drive/MyDrive/BlueCARES Drone Data/SGC_CEB_202302221159E_56m_20230308.tif"          # large input GeoTIFF
OUTPUT_PATH  = "/content/drive/MyDrive/BlueCARES_Seagrass/inference_results/SGC_CEB_202302221159E_56m_20230308_mask.tif"      # binary mask output
OVERLAY_PATH = "/content/drive/MyDrive/BlueCARES_Seagrass/inference_results/SGC_CEB_202302221159E_56m_20230308.png"              # QC visualisation

TILE_SIZE    = 512    # must match training size
OVERLAP      = 64     # px overlap between tiles (64–128 recommended)
BATCH_SIZE   = 4      # tiles per GPU batch (reduce if OOM)
THRESHOLD    = 0.5    # foreground probability threshold
SAVE_PROBMAP = False  # True → also write a float32 probability GeoTIFF
# ============================================================

# --- Load model ---
model = load_model(MODEL_PATH)

# --- Load orthomosaic ---
image, meta = load_orthomosaic(INPUT_PATH)

# --- Sliding-window inference ---
pred_mask, prob_map = sliding_window_inference(
    model,
    image,
    tile_size  = TILE_SIZE,
    overlap    = OVERLAP,
    batch_size = BATCH_SIZE,
    threshold  = THRESHOLD,
)

# --- Save outputs ---
save_mask(pred_mask, OUTPUT_PATH, meta)
if SAVE_PROBMAP:
    save_probability_map(prob_map, OUTPUT_PATH, meta)

# --- Visualise ---
visualise_result(
    image, pred_mask,
    prob_map  = prob_map if SAVE_PROBMAP else None,
    save_path = OVERLAY_PATH,
)

# --- Clean up GPU memory ---
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"\n[DONE] Inference complete. Mask shape: {pred_mask.shape}")

[INFO] Loading model from: /content/drive/MyDrive/BlueCARES_Seagrass/segmentation/training_results/best_model_epoch_038.pth
[INFO] Model loaded on cpu
[INFO] Loaded via rasterio — shape: (20587, 11789, 3), dtype: uint8
[INFO] Final orthomosaic shape: (20587, 11789, 3)
[INFO] Orthomosaic : 20587 x 11789 px
[INFO] Tile 512px | Overlap 64px | Batch 4
[INFO] Total tiles : 1242


Inference:  16%|█▌        | 50/311 [3:27:15<17:58:14, 247.87s/it]